# Metadata Grounding versus Prompt Engineering in Compute-Constrained Vision-Language Models

Reproduction code for the paper *AI-Enabled Product Demonstration Content for Direct
Selling: Metadata Grounding versus Prompt Engineering in Compute-Constrained
Vision-Language Models*.

---

## What this notebook does

Generates product demonstration scripts for 150 products from the Amazon Berkeley
Objects catalogue, using two open-weight vision-language models under three prompt
conditions, then scores the output against the catalogue record. 900 generations
total, fully crossed and paired within product.

## How to run it

| Goal | Set | Runtime |
|---|---|---|
| Reproduce tables and figures from released outputs | `RUN_GENERATION = False` | ~2 min, CPU |
| Regenerate everything from scratch | `RUN_GENERATION = True` | ~1.5 h, one T4 |

Part 3 always re-scores from the raw JSONL, so changing a metric never requires
regeneration.

## Where each paper artifact comes from

| Paper | Produced by |
|---|---|
| Table 1 (main results) | Part 7 |
| Table 2 (paired effect sizes) | Part 5 |
| Table 3 (threshold sensitivity) | Part 7 |
| Table 4 (sample composition) | Part 7 |
| Figure 1 (coverage) | Part 6 |
| Figure 2 (well-formedness) | Part 6 |
| Figure 3 (output length) | Part 6 |
| Figure 4 (paired scatter) | Part 6 |
| Figure 5 (effect sizes) | Part 6 |
| Section 6.6 (category robustness) | Part 4.2 |
| Section 8 (metric audit) | Part 4.3 |

## Data licence

Amazon Berkeley Objects is released under CC BY-NC 4.0. Research use only; commercial
use is prohibited. Credit for the data, including all images and 3D models, is due to
Matthieu Guillaumin, Thomas Dideriksen, Kenan Deng and Himanshu Arora (Amazon.com),
and Jasmine Collins and Jitendra Malik (UC Berkeley). This repository redistributes no
ABO content; it releases the sampled item identifiers and the code that reconstructs
the sample.

Code in this repository is MIT licensed. Derived artifacts inherit the non-commercial
restriction of the source data.

## Part 0 — Configuration

In [ ]:
CONFIG = {
    "n_products": 150,
    "seed": 20260719,
    "listings_shards": ["0", "1", "2"],
    "max_new_tokens": 150,
    "usable_min_trigram": 0.6,
    "models": [
        # revision: pin to a commit hash for exact reproduction. None resolves to the
        # repository default, which is what the published run used. See paper Sec. 5.3.
        {"key": "smolvlm-500m", "repo": "HuggingFaceTB/SmolVLM-500M-Instruct", "revision": None},
        {"key": "qwen2vl-2b",   "repo": "Qwen/Qwen2-VL-2B-Instruct",          "revision": None},
    ],
    "conditions": ["image_meta", "image_only", "image_only_scaffold"],
    "target_types": ["HOME", "KITCHEN", "STORAGE", "LAMP", "CHAIR", "TABLE",
                     "BAG", "SHOE", "WATCH", "JEWELRY", "CUP", "BOTTLE", "PILLOW"],
}

RUN_GENERATION = False   # True only to regenerate the 900 outputs on a GPU
NEED_IMAGES = RUN_GENERATION

S3 = "https://amazon-berkeley-objects.s3.amazonaws.com"

In [ ]:
# !pip -q install "transformers==4.51.3" "accelerate==1.6.0" "huggingface-hub==0.36.2" pillow pandas numpy scipy matplotlib tqdm requests

In [ ]:
import gzip, json, os, platform, random, re, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

# Paths. On Colab, results persist to Drive; locally they sit beside the notebook.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/abo-vlm-grounding")
    WORK = Path("/content/work")
except Exception:
    ROOT = Path("./results")
    WORK = Path("./.cache")

for d in (ROOT, WORK / "images", ROOT / "figures"):
    d.mkdir(parents=True, exist_ok=True)

MANIFEST = ROOT / "manifest.csv"
FIG = ROOT / "figures"

random.seed(CONFIG["seed"]); np.random.seed(CONFIG["seed"])
print("results ->", ROOT.resolve())
print("existing:", sorted(p.name for p in ROOT.iterdir()) or "(empty)")

## Part 1 — Sample construction

Skipped if `manifest.csv` is present. Reproduces the exact 150-product sample from the
fixed seed. Published run: 27,696 listings from three shards, 4,830 eligible across
50 product types, 150 sampled spanning 40 types.

In [ ]:
def fetch(url, dest, retries=3):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    for i in range(retries):
        try:
            with requests.get(url, stream=True, timeout=180) as r:
                r.raise_for_status()
                tmp = dest.with_suffix(dest.suffix + ".part")
                with open(tmp, "wb") as fh:
                    for chunk in r.iter_content(1 << 20):
                        fh.write(chunk)
                tmp.rename(dest)
            return dest
        except Exception as e:
            if i == retries - 1:
                raise
            print(f"  retry {i+1}: {e}"); time.sleep(2 * (i + 1))
    return dest


def en_value(field):
    """ABO text fields are lists of {language_tag, value}. Prefer English."""
    if not isinstance(field, list):
        return None
    for e in field:
        if isinstance(e, dict) and str(e.get("language_tag", "")).startswith("en"):
            return e.get("value")
    for e in field:
        if isinstance(e, dict) and e.get("value"):
            return e["value"]
    return None


def parse_listing(rec):
    return {
        "item_id": rec.get("item_id"),
        "main_image_id": rec.get("main_image_id"),
        "product_type": (rec.get("product_type") or [{}])[0].get("value"),
        "item_name": en_value(rec.get("item_name")),
        "brand": en_value(rec.get("brand")),
        "color": en_value(rec.get("color")),
        "material": en_value(rec.get("material")),
        "style": en_value(rec.get("style")),
    }


if MANIFEST.exists():
    sample = pd.read_csv(MANIFEST)
    print(f"manifest found: {len(sample)} products, "
          f"{sample.product_type.nunique()} product types")
else:
    rows = []
    for shard in CONFIG["listings_shards"]:
        p = fetch(f"{S3}/listings/metadata/listings_{shard}.json.gz",
                  WORK / f"listings_{shard}.json.gz")
        with gzip.open(p, "rt", encoding="utf-8") as fh:
            rows += [parse_listing(json.loads(l)) for l in fh]
    listings = pd.DataFrame(rows)
    print(f"loaded {len(listings):,} listings from {len(CONFIG['listings_shards'])} shards")

    CORE = ["brand", "color", "material"]
    pool = listings[
        listings.main_image_id.notna() & listings.item_name.notna()
        & listings.product_type.notna() & (listings[CORE].notna().sum(axis=1) >= 2)
    ].copy()
    pool = pool[pool.product_type.str.upper()
                    .str.contains("|".join(CONFIG["target_types"]), na=False)]
    print(f"eligible pool: {len(pool):,} products across {pool.product_type.nunique()} types")

    n, parts = CONFIG["n_products"], []
    for _, grp in pool.groupby("product_type"):
        take = max(1, round(n * len(grp) / len(pool)))
        parts.append(grp.sample(min(take, len(grp)), random_state=CONFIG["seed"]))
    sample = (pd.concat(parts).sample(frac=1, random_state=CONFIG["seed"])
                .head(n).reset_index(drop=True))
    sample.to_csv(MANIFEST, index=False)
    sample[["item_id"]].to_csv(ROOT / "item_ids.csv", index=False)
    print(f"sampled {len(sample)} products across {sample.product_type.nunique()} types")

In [ ]:
image_files = {}
if NEED_IMAGES:
    idx = fetch(f"{S3}/images/metadata/images.csv.gz", WORK / "images.csv.gz")
    id_to_path = dict(pd.read_csv(idx)[["image_id", "path"]].values)
    for _, row in tqdm(sample.iterrows(), total=len(sample), desc="images"):
        rel = id_to_path.get(row["main_image_id"])
        if rel is None:
            continue
        dest = WORK / "images" / f"{row['item_id']}.jpg"
        try:
            fetch(f"{S3}/images/small/{rel}", dest)
            image_files[row["item_id"]] = dest
        except Exception as e:
            print(f"  skip {row['item_id']}: {e}")
    sample = sample[sample.item_id.isin(image_files)].reset_index(drop=True)
    print(f"{len(sample)} products with images")
else:
    print("image download skipped (NEED_IMAGES is False)")

## Part 2 — Generation (GPU)

Resumable: an interrupted run resumes from the completed keys. Greedy decoding, so a
generation is a deterministic function of model revision, library version and input.

In [ ]:
SYSTEM = ("You are helping a direct-selling distributor demonstrate a product to a "
          "customer. Describe only what you can verify. Do not invent specifications.")

TASK = ("Write a short spoken product demonstration script (about 80 words). "
        "Mention what the product is, its key visible features, and one practical benefit.")

SCAFFOLD = ("Write a spoken product demonstration script of 60-100 words.\n"
            "Structure it in exactly three sentences:\n"
            "1. What the product is.\n"
            "2. Two features you can see in the image.\n"
            "3. One practical benefit for the customer.\n"
            "Write the script only. Do not write a label or a single word.")


def build_prompt(row, condition):
    if condition == "image_only":
        return TASK
    if condition == "image_only_scaffold":
        return SCAFFOLD
    facts = [f"Product type: {row['product_type']}"]
    for key in ["item_name", "brand", "color", "material", "style"]:
        v = row.get(key)
        if pd.notna(v) and v:
            facts.append(f"{key.replace('_',' ').title()}: {v}")
    return f"{TASK}\n\nVerified product information:\n" + "\n".join(facts)


def done_keys(path):
    keys = set()
    if Path(path).exists():
        for line in open(path):
            try:
                r = json.loads(line)
                keys.add((r["item_id"], r["condition"]))
            except Exception:
                pass
    return keys


if RUN_GENERATION:
    import torch
    from PIL import Image
    from transformers import AutoModelForVision2Seq, AutoProcessor

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
    print("device:", DEVICE)

    for spec in CONFIG["models"]:
        out_path = ROOT / f"gen_{spec['key']}.jsonl"
        completed = done_keys(out_path)
        todo = [(r, c) for _, r in sample.iterrows()
                for c in CONFIG["conditions"] if (r["item_id"], c) not in completed]
        if not todo:
            print(f"{spec['key']}: complete"); continue

        kw = {"revision": spec["revision"]} if spec.get("revision") else {}
        processor = AutoProcessor.from_pretrained(spec["repo"], **kw)
        model = AutoModelForVision2Seq.from_pretrained(
            spec["repo"], torch_dtype=DTYPE, device_map=DEVICE,
            attn_implementation="eager", **kw).eval()
        resolved = getattr(model.config, "_commit_hash", None) or spec.get("revision")

        with open(out_path, "a") as sink:
            for row, cond in tqdm(todo, desc=spec["key"]):
                img = Image.open(image_files[row["item_id"]]).convert("RGB")
                img.thumbnail((512, 512))
                msgs = [{"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
                        {"role": "user", "content": [{"type": "image"},
                         {"type": "text", "text": build_prompt(row, cond)}]}]
                text = processor.apply_chat_template(msgs, add_generation_prompt=True)
                inputs = processor(text=[text], images=[img], return_tensors="pt").to(DEVICE)
                t0 = time.time()
                with torch.inference_mode():
                    ids = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"],
                                         do_sample=False)
                trimmed = ids[:, inputs["input_ids"].shape[1]:]
                sink.write(json.dumps({
                    "item_id": row["item_id"], "model": spec["key"], "condition": cond,
                    "output": processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip(),
                    "seconds": round(time.time() - t0, 2),
                    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
                    "revision": resolved,
                }) + "\n")
                sink.flush()
        del model
        torch.cuda.empty_cache()
else:
    print("RUN_GENERATION is False — using released gen_*.jsonl")

## Part 3 — Scoring

Rebuilt from raw JSONL every run. Metric definitions live here and nowhere else.

**Metric revision history** (paper Sec. 5.4). Two earlier definitions of output
usability were discarded: a raw word-count threshold, and a minimum sentence count.
Both penalise compliance with the scaffolded instruction, which asks for shorter
output. The final definition uses only prompt echo and repetition, which are
independent of instructed length. Attribute coverage moved by at most 0.030 across
all three definitions.

In [ ]:
COLORS = {"black","white","grey","gray","silver","gold","beige","brown","red","blue",
          "navy","green","yellow","orange","pink","purple","ivory","cream"}
MATERIALS = {"wood","wooden","metal","steel","aluminum","aluminium","plastic","glass",
             "leather","cotton","linen","silk","ceramic","bamboo","marble","rubber",
             "polyester","velvet","brass","copper"}
PROMPT_MARKERS = ["two features","one practical benefit","60-100 words","product:",
                  "write a spoken","you can see in the image"]
SCORED_ATTRS = ["brand", "color", "material", "product_type"]


def norm(t):
    return re.sub(r"[^a-z0-9 ]", " ", str(t).lower())


def rep_ratio(text, n=3):
    w = str(text).lower().split()
    if len(w) < n + 1:
        return 0.0
    g = [tuple(w[i:i+n]) for i in range(len(w)-n+1)]
    return len(set(g)) / len(g)


def score_record(rec, gt):
    """Rule-based scoring of one generation against its catalogue record."""
    text = norm(rec["output"])
    tokens = set(text.split())

    available = covered = 0
    for a in SCORED_ATTRS:
        v = gt.get(a)
        if pd.isna(v) or not v:
            continue
        available += 1
        if any(w in text for w in norm(v).split() if len(w) > 2):
            covered += 1

    supported = set(norm(f"{gt.get('color','')} {gt.get('material','')} "
                         f"{gt.get('item_name','')}").split())
    return {
        "coverage": covered / available if available else np.nan,
        "n_attrs": available,
        "unsupported_terms": len((tokens & (COLORS | MATERIALS)) - supported),
        "words": len(text.split()),
        "echo": any(m in str(rec["output"]).lower() for m in PROMPT_MARKERS),
        "reps": rep_ratio(rec["output"]),
    }


gt_map = pd.read_csv(MANIFEST).set_index("item_id").to_dict("index")
records = []
for spec in CONFIG["models"]:
    p = ROOT / f"gen_{spec['key']}.jsonl"
    if not p.exists():
        print("missing:", p.name); continue
    for line in open(p):
        rec = json.loads(line)
        gt = gt_map.get(rec["item_id"])
        if gt:
            records.append({**rec, **score_record(rec, gt)})

df = pd.DataFrame(records)
df["usable"] = ~(df.echo | (df.reps < CONFIG["usable_min_trigram"]))
df["unsup_any"] = df.unsupported_terms > 0
df.to_csv(ROOT / "scored.csv", index=False)

usable = df[df.usable]
ORDER = [c for c in CONFIG["conditions"] if c in set(df.condition)]
MODELS = sorted(df.model.unique())
ML = {"qwen2vl-2b": "Qwen2-VL-2B", "smolvlm-500m": "SmolVLM-500M"}
ML = {m: ML.get(m, m) for m in MODELS}
print(f"{len(df)} scored records, {df.item_id.nunique()} products")

## Part 4 — Validation

In [ ]:
print("=== 4.1 INTEGRITY ===")
print(df.groupby(["model", "condition"]).size().unstack().to_string(), "\n")
print("duplicate (model, condition, item):", df.duplicated(["model","condition","item_id"]).sum())
print("missing coverage:", int(df.coverage.isna().sum()))
print("accelerators:", sorted(set(df.gpu.dropna())))
print(f"attributes per product: mean {df.n_attrs.mean():.2f}, "
      f"min {df.n_attrs.min()}, max {df.n_attrs.max()}")

In [ ]:
print("\n=== 4.2 CATEGORY ROBUSTNESS (paper Sec. 6.6) ===")
man = pd.read_csv(MANIFEST)[["item_id", "product_type"]]
dfc = usable.merge(man, on="item_id", how="left")
biggest = man.product_type.value_counts().index[0]
print(f"largest category: {biggest} "
      f"({(man.product_type == biggest).mean()*100:.1f}% of sample)\n")
for m in MODELS:
    for label, sub in [("all", dfc), (f"excl. {biggest}", dfc[dfc.product_type != biggest])]:
        a = sub[(sub.model == m) & (sub.condition == "image_meta")].set_index("item_id")
        b = sub[(sub.model == m) & (sub.condition == "image_only")].set_index("item_id")
        k = a.index.intersection(b.index)
        if len(k) >= 5:
            print(f"  {ML[m]:14s} {label:16s} n={len(k):3d} "
                  f"delta={a.loc[k,'coverage'].mean() - b.loc[k,'coverage'].mean():+.4f}")

In [ ]:
print("\n=== 4.3 MANUAL AUDIT EXPORT (paper Sec. 8) ===")
# The automatic coverage metric is string matching. This exports a stratified sample
# for blind human rating, which validates the metric against human judgement.
parts = [g.sample(min(8, len(g)), random_state=CONFIG["seed"])
         for _, g in usable.groupby(["model", "condition"])]
audit = pd.concat(parts, ignore_index=True).merge(
    pd.read_csv(MANIFEST)[["item_id","item_name","brand","color","material","product_type"]],
    on="item_id", how="left")
audit = audit.sample(frac=1, random_state=CONFIG["seed"]).reset_index(drop=True)
audit["rating_id"] = [f"R{i+1:03d}" for i in range(len(audit))]

audit[["rating_id","item_id","model","condition","coverage","unsup_any"]].to_csv(
    ROOT / "audit_unblinding_key.csv", index=False)
blind = audit[["rating_id","output","item_name","brand","color","material","product_type"]].copy()
for c in ["attributes_present","attributes_available","unsupported_claim","notes"]:
    blind[c] = ""
for r in ("A","B"):
    blind.to_csv(ROOT / f"audit_rating_sheet_{r}.csv", index=False)
print(f"wrote {len(blind)} blinded rows for two raters, plus the unblinding key")

## Part 5 — Statistics (paper Table 2)

In [ ]:
from scipy.stats import wilcoxon


def boot_ci(v, n=2000, seed=0):
    v = np.asarray(v, float); v = v[~np.isnan(v)]
    if len(v) < 2:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    return np.percentile(v[rng.integers(0, len(v), (n, len(v)))].mean(axis=1), [2.5, 97.5])


PAIRS = [("image_meta", "image_only", "Metadata grounding"),
         ("image_only", "image_only_scaffold", "Prompt scaffolding")]
rows = []
for m in MODELS:
    for c1, c2, lab in PAIRS:
        if c1 not in ORDER or c2 not in ORDER:
            continue
        a = usable[(usable.model == m) & (usable.condition == c1)].set_index("item_id")
        b = usable[(usable.model == m) & (usable.condition == c2)].set_index("item_id")
        k = a.index.intersection(b.index)
        if len(k) < 5:
            continue
        diff = (a.loc[k, "coverage"] - b.loc[k, "coverage"]).values
        sd = diff.std(ddof=1)
        if not np.isfinite(sd) or sd == 0:
            continue
        rng = np.random.default_rng(1)
        bs = diff[rng.integers(0, len(diff), (2000, len(diff)))]
        ds = bs.mean(axis=1) / bs.std(axis=1, ddof=1)
        lo, hi = np.percentile(ds, [2.5, 97.5])
        _, p = wilcoxon(a.loc[k, "coverage"], b.loc[k, "coverage"])
        rows.append({"model": ML[m], "intervention": lab, "n": len(k),
                     "delta": diff.mean(), "d": diff.mean()/sd,
                     "ci_lo": lo, "ci_hi": hi, "p": p})

eff = pd.DataFrame(rows)
eff["p_bonferroni"] = (eff.p * len(eff)).clip(upper=1.0)
eff.to_csv(ROOT / "table2_effects.csv", index=False)
print(eff.round(4).to_string(index=False))

## Part 6 — Figures

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({"font.family": "serif", "font.size": 9, "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "savefig.dpi": 300, "savefig.bbox": "tight"})
GREY, LIGHT = "#4d4d4d", "#a6a6a6"
LAB = {"image_meta": "Image +\nmetadata", "image_only": "Image\nonly",
       "image_only_scaffold": "Image only +\nscaffold"}
SHORT = {"image_meta": "Img+meta", "image_only": "Img only",
         "image_only_scaffold": "Img+scaf"}


def save(fig, name):
    for ext in ("png", "tiff"):
        fig.savefig(FIG / f"{name}.{ext}")
    plt.show()


# Figure 1 — attribute coverage
fig, ax = plt.subplots(figsize=(5.2, 3.0))
w, x = 0.36, np.arange(len(ORDER))
for i, m in enumerate(MODELS):
    mu, lo, hi = [], [], []
    for c in ORDER:
        v = usable[(usable.model == m) & (usable.condition == c)]["coverage"]
        a_, b_ = boot_ci(v)
        mu.append(v.mean()); lo.append(v.mean() - a_); hi.append(b_ - v.mean())
    ax.bar(x + (i - 0.5) * w, mu, w, yerr=[lo, hi], capsize=3, label=ML[m],
           color=[GREY, LIGHT][i % 2], edgecolor="black", linewidth=0.6)
ax.set_xticks(x); ax.set_xticklabels([LAB[c] for c in ORDER])
ax.set_ylabel("Attribute coverage"); ax.set_ylim(0, 0.8)
ax.legend(frameon=False); ax.grid(axis="y", linewidth=0.4, alpha=0.4); ax.set_axisbelow(True)
save(fig, "fig1_coverage")

In [ ]:
# Figure 2 — output well-formedness
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.5))
for ax, (col, ylab, ylim) in zip(axes, [("usable", "Usable output rate", (0, 1.05)),
                                        ("echo", "Prompt echo rate", (0, 0.4)),
                                        ("reps", "Distinct-trigram ratio", (0.6, 1.02))]):
    for i, m in enumerate(MODELS):
        y = [df[(df.model == m) & (df.condition == c)][col].mean() for c in ORDER]
        ax.plot(range(len(ORDER)), y, marker=["o", "s"][i % 2], color=[GREY, LIGHT][i % 2],
                label=ML[m], linewidth=1.4, markersize=5)
    ax.set_xticks(range(len(ORDER)))
    ax.set_xticklabels([SHORT[c] for c in ORDER], rotation=20)
    ax.set_ylabel(ylab); ax.set_ylim(*ylim)
    ax.grid(axis="y", linewidth=0.4, alpha=0.4); ax.set_axisbelow(True)
axes[0].legend(frameon=False, loc="lower left")
plt.tight_layout(); save(fig, "fig2_wellformedness")

In [ ]:
# Figure 3 — output length
fig, ax = plt.subplots(figsize=(5.6, 3.0))
data, pos, cols = [], [], []
for j, c in enumerate(ORDER):
    for i, m in enumerate(MODELS):
        data.append(df[(df.model == m) & (df.condition == c)]["words"].values)
        pos.append(j * 3 + i); cols.append([GREY, LIGHT][i % 2])
bp = ax.boxplot(data, positions=pos, widths=0.75, patch_artist=True, showfliers=False,
                medianprops=dict(color="black", linewidth=1.2))
for pch, cc in zip(bp["boxes"], cols):
    pch.set_facecolor(cc); pch.set_edgecolor("black"); pch.set_linewidth(0.6)
ax.set_xticks([j * 3 + 0.5 for j in range(len(ORDER))])
ax.set_xticklabels([LAB[c] for c in ORDER]); ax.set_ylabel("Output length (words)")
ax.legend([plt.Rectangle((0,0),1,1, facecolor=cc, edgecolor="black", linewidth=0.6)
           for cc in (GREY, LIGHT)], [ML[m] for m in MODELS], frameon=False)
ax.grid(axis="y", linewidth=0.4, alpha=0.4); ax.set_axisbelow(True)
save(fig, "fig3_length")

In [ ]:
# Figure 4 — paired scatter, length-matched comparison
target = "qwen2vl-2b" if "qwen2vl-2b" in MODELS else MODELS[0]
a = usable[(usable.model == target) & (usable.condition == "image_meta")].set_index("item_id")
b = usable[(usable.model == target) & (usable.condition == "image_only")].set_index("item_id")
k = a.index.intersection(b.index)
fig, ax = plt.subplots(figsize=(3.4, 3.4))
jit = np.random.default_rng(0).normal(0, 0.012, len(k))
ax.scatter(a.loc[k, "coverage"] + jit, b.loc[k, "coverage"] + jit, s=14, alpha=0.5,
           color=GREY, edgecolor="none")
ax.plot([0, 1], [0, 1], color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Coverage, image + metadata"); ax.set_ylabel("Coverage, image only")
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05); ax.set_aspect("equal")
ax.set_title(f"{ML[target]}: n={len(k)}, "
             f"{(b.loc[k,'coverage'] < a.loc[k,'coverage']).mean():.0%} below diagonal",
             fontsize=8)
save(fig, "fig4_paired")

In [ ]:
# Figure 5 — effect sizes
ep = eff.dropna(subset=["d", "ci_lo", "ci_hi"]).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(5.2, 2.6))
y = np.arange(len(ep))[::-1]
ax.errorbar(ep.d, y, xerr=[ep.d - ep.ci_lo, ep.ci_hi - ep.d], fmt="o", color=GREY,
            capsize=3, markersize=5, linewidth=1.2)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_yticks(y)
ax.set_yticklabels([f"{r.model}\n{r.intervention}" for r in ep.itertuples()], fontsize=7.5)
ax.set_xlabel("Cohen's d (paired), 95% bootstrap CI")
ax.grid(axis="x", linewidth=0.4, alpha=0.4); ax.set_axisbelow(True)
save(fig, "fig5_effectsizes")

## Part 7 — Tables and provenance

In [ ]:
t1 = (df.groupby(["model", "condition"])
        .agg(n=("output", "size"), usable_rate=("usable", "mean"),
             echo_rate=("echo", "mean"), distinct_trigram=("reps", "mean"),
             words=("words", "mean"), sec_median=("seconds", "median"))
        .join(usable.groupby(["model", "condition"])
                .agg(usable_n=("coverage", "size"), coverage=("coverage", "mean"),
                     coverage_sd=("coverage", "std"), unsup_any=("unsup_any", "mean")))
        .round(3).reset_index())
t1["condition"] = pd.Categorical(t1.condition, ORDER, ordered=True)
t1 = t1.sort_values(["model", "condition"])
t1.to_csv(ROOT / "table1_main.csv", index=False)

t3 = pd.concat([df.assign(c=df.words >= t).groupby(["model","condition"])["c"]
                  .mean().rename(f"ge_{t}w") for t in (10, 20, 30, 50, 70)],
               axis=1).round(3).reset_index()
t3.to_csv(ROOT / "table3_threshold_sensitivity.csv", index=False)

t4 = pd.read_csv(MANIFEST).product_type.value_counts().rename("n").to_frame()
t4["pct"] = (t4.n / t4.n.sum() * 100).round(1)
t4.reset_index(names="product_type").to_csv(ROOT / "table4_sample_composition.csv", index=False)

with open(ROOT / "provenance.json", "w") as fh:
    json.dump({
        "config": CONFIG,
        "n_records": int(len(df)),
        "n_products": int(df.item_id.nunique()),
        "product_types_in_sample": int(pd.read_csv(MANIFEST).product_type.nunique()),
        "accelerators": sorted(set(df.gpu.dropna())),
        "revisions": sorted({r for r in df.get("revision", pd.Series(dtype=str)).dropna()}),
        "python": sys.version, "platform": platform.platform(),
        "pandas": pd.__version__, "numpy": np.__version__,
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
    }, fh, indent=2, default=str)

print(t1.to_string(index=False))
print("\nfigures:", FIG)
print("tables and provenance:", ROOT)